In [ ]:
%cd ../..


In [ ]:
from scripts.globals import XGB_PIPELINE_DIR
import pandas as pd
import matplotlib.pyplot as plt

import matplotlib.colors as mcolors
import seaborn as sns
pd.set_option('display.max_columns', None)


In [ ]:
class DropRowsCache:
    total_dropped = 0
    
    @classmethod
    def drop_rows_and_report(cls, df, condition):
        filtered_df = df.loc[condition]
        num_dropped = filtered_df.shape[0]
        cls.total_dropped += num_dropped
        df_dropped = df.drop(filtered_df.index)
        
        print(f"Number of dropped rows: {num_dropped}")
        print(f"Total number of dropped rows: {cls.total_dropped}")
        
        return df_dropped

In [ ]:
df = pd.read_csv(f"{XGB_PIPELINE_DIR}/1_find_matches_for_clash.csv")

df.head()

In [ ]:
# initiating drop rows method with cache
cache = DropRowsCache()

# dropping rows with low structural stability (energy close to 0)
cond_energy = df['true_energy'] > -5
df = cache.drop_rows_and_report(df, cond_energy)

# dropping rows where true seed base pairs doesn't match true seed type
cond_6mer = (df["true_seed_basepairs"] <= 4) & (df["true_seed_type"] == "6-mer")
cond_7mer = (df["true_seed_basepairs"] <= 4) & (df["true_seed_type"] == "7-mer")
cond_8mer = (df["true_seed_basepairs"] <= 4) & (df["true_seed_type"] == "8-mer")

df = cache.drop_rows_and_report(df, cond_6mer)
df = cache.drop_rows_and_report(df, cond_7mer)
df = cache.drop_rows_and_report(df, cond_8mer)

# dropping rows where true number of bp is lower than 5
cond_true_numbp = df['true_num_basepairs'] < 5
df = cache.drop_rows_and_report(df, cond_true_numbp)

# dropping rows where predicted number of bp is lower than 5
cond_pred_numbp = df['pred_num_basepairs'] < 5
df = cache.drop_rows_and_report(df, cond_pred_numbp)

# dropping rows where CLASH sequence not matching ENSEMBL 60

In [ ]:
# these indices are generated in preprocess_clash file
rows_to_drop = [4508, 6518, 8028, 10039, 15835, 17039]
df.drop(index=rows_to_drop, inplace=True)

# visualizing the df

In [ ]:
subdf = df[["pred_seed_basepairs", "true_seed_basepairs"]]

pivot_table = subdf.pivot_table(index='pred_seed_basepairs', columns='true_seed_basepairs', aggfunc=len, fill_value=0)
pivot_table = pivot_table[::-1]

# Create a logarithmic color scaling norm
norm = mcolors.LogNorm(vmin=pivot_table.values.min(), vmax=pivot_table.values.max())

sns.heatmap(pivot_table, fmt='d', cmap='Greens', norm=norm, annot=True)
plt.show()

# dropping cells where the difference between pred & true basepairs is bigger than one


In [ ]:
df = df[(df.pred_seed_basepairs - df.true_seed_basepairs).abs() <= 1]

In [ ]:
subdf2 = df[["pred_seed_basepairs", "true_seed_basepairs"]]

pivot_table = subdf2.pivot_table(index='pred_seed_basepairs', columns='true_seed_basepairs', aggfunc=len, fill_value=0)
pivot_table = pivot_table[::-1]

sns.heatmap(pivot_table, annot=True, fmt='d', cmap='Greens')
plt.show()

In [ ]:
# export to csv
cols_to_keep = ["id", 'mrna_start', 'mrna_end', 'pred_energy', 'mirna_start', 'mirna_end',
                'mirna_dot_bracket_5to3', 'mirna_sequence', "mirna_accession", "mre_region", "enst", "full_sequence_of_transcript",
                "extended_mrna_start", "extended_mrna_end", "extended_mrna_sequence", "clash_mrna_start", "clash_mrna_end", "mre_start", "mre_end"]

df[cols_to_keep].to_csv(f"{XGB_PIPELINE_DIR}/2_positive_data.csv", index=False)
